M**odule 9: RAG Evaluation & Observability**

Step 1: Imports and Environment

In [1]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv

Step 2: Load Vector Store and Build RAG Chain

In [2]:
from langsmith import Client
from langsmith.evaluation import evaluate

In [3]:
# Create a LangSmith client
client = Client()
print('LangSmith evaluation tools ready.')

LangSmith evaluation tools ready.


Create a Dataset and Run Evaluation

* Create a Dataset

In [4]:
# Create a dataset in LangSmith
dataset_name = 'rag_eval_dataset'

# Create dataset if it doesn't exist
try:
    dataset = client.create_dataset(
        dataset_name, 
        description='Simple RAG evaluation questions'
    )
    print(f'Created dataset: {dataset_name}')

except Exception as e:
    # If it already exists, retrieve it
    datasets = list(client.list_datasets(dataset_name=dataset_name))
    dataset = datasets[0]
    print(f'Dataset already exists, using: {dataset_name}')

#print(f'Dataset object: {dataset}') 
print('Dataset object:', dataset)   

Dataset already exists, using: rag_eval_dataset
Dataset object: name='rag_eval_dataset' description='Simple RAG evaluation questions' data_type=<DataType.kv: 'kv'> id=UUID('f6eeba24-c448-4e08-82e0-e6a4761f2847') created_at=datetime.datetime(2026, 9, 9, 11, 20, 31, 157051, tzinfo=datetime.timezone.utc) modified_at=datetime.datetime(2026, 9, 9, 11, 20, 31, 157051, tzinfo=datetime.timezone.utc) example_count=15 session_count=0 last_session_start_time=None


Add Examples to Dataset

In [5]:
# # Define example questions
examples = [
    'What are the common crop diseases and their control methods?',
    'What are the top causes of death in Nigeria?',
    'What are the symptoms of Cassava Mosaic Disease?',
    'What is Mastitis and how is it managed in dairy animals?',
    "What product has the comment 'Spacious and strong'?",
]

# # Prepare inputs and outputs for examples
# inputs = [{'question': q} for q in examples]
# outputs = [{'answer': ''} for _ in examples]

# # Add examples to the dataset
# client.create_examples(
#     inputs=inputs,
#     outputs=outputs,
#     dataset_id=dataset.id
# )

# print('Examples added to dataset.')

In [6]:
# # Use the examples list (strings) from earlier
# for q in examples:
#     print(f'Question: {q}')
#     ans = rag_chain.invoke(q)
#     print(f'Answer: {ans}\n---')

In [7]:
# # Debug retrieval
# test_q = 'What are the top causes of death in Nigeria?'
# docs = retriever.invoke(test_q)

# print(f'Retrieved {len(docs)} documents for question: "{test_q}"')
# for doc in docs:
#     print('-', doc.page_content[:200])

Define Target Function and Run Evaluation

In [8]:
# # Define examples as a list of dictionaries
# examples_data = [
#     {'question': q, 'answer': ''}
#     for q in examples
# ]

# # Define a wrapper that expects a dict with 'question'
# def target_fn(example: dict) -> str:
    
#     return rag_chain.invoke(example['question'])

# # Run evaluation
# results = evaluate(
#     target_fn,
#     data=examples_data,
#     evaluators=[],
#     experiment_prefix='basic_rag_eval'
# )

# print('Evaluation run completed.')
# print(results)

Import Document Loaders and Splitter

In [9]:
from langchain_community.document_loaders import PyPDFLoader, BSHTMLLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

Load Only Relevant Documents

In [10]:
# Load only health and agriculture documents
health_pdf = PyPDFLoader('../../../04_data_ingestion_document_processing/data/nigeria_health_diseases_and_prevention.pdf').load()
crop_pdf = PyPDFLoader('../../../04_data_ingestion_document_processing/data/crop_disease.pdf').load()
agri_html = BSHTMLLoader('../../../04_data_ingestion_document_processing/data/agriculture.html', open_encoding='utf-8', bs_kwargs={'features': 'html.parser'}).load()
agri_txt = TextLoader('../../../04_data_ingestion_document_processing/data/agriculture.txt', encoding='utf-8').load()

all_docs = health_pdf + crop_pdf + agri_html + agri_txt
print(f'Loaded {len(all_docs)} relevant documents.')

Loaded 37 relevant documents.


Split and Add Metadata

In [11]:
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)

chunks = splitter.split_documents(all_docs)

for i, chunk in enumerate(chunks):
    source = chunk.metadata.get('source', '')
    file_name = source.split('\\')[-1] if '\\' in source else source.split('/')[-1]
    chunk.metadata['file_name'] = file_name
    chunk.metadata['doc_type'] = (
        'pdf' if file_name.endswith('.pdf')
        else 'html' if file_name.endswith('.html')
        else 'txt'
    )
    chunk.metadata['language'] = 'English'
    chunk.metadata['chunk_id'] = f'{file_name}_{i+1:03d}'

print(f'Created {len(chunks)} clean chunks.')

Created 185 clean chunks.


Create a New Clean Vector Store

In [12]:
# Create new embeddings and vectorstore
embeddings = OpenAIEmbeddings()

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory='../../../chroma_db_clean'
)

# Create retriever from clean store
retriever = vectorstore.as_retriever(search_kwargs={'k': 4})
print('Clean vector store ready.')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Clean vector store ready.


Build the RAG Chain

In [13]:
# LLM and prompt
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
rag_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant. Answer the question using only the provided context. If you don\'t know, say you don\'t know.'),
    ('human', 'Context:\n{context}\n\nQuestion: {question}')
])

def format_docs(docs):
    return '\n\n'.join(doc.page_content for doc in docs)

rag_chain = (
    {
        'context': retriever | format_docs,
        'question': RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

print('RAG chain rebuilt with clean retriever.')

RAG chain rebuilt with clean retriever.


Test manual evaluation

In [14]:
for q in examples:
    print(f'Question: {q}')
    ans = rag_chain.invoke(q)
    print(f'Answer: {ans}\n---')

Question: What are the common crop diseases and their control methods?


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Answer: The common crop disease mentioned is Cassava Mosaic Disease. The affected crop is cassava, and the symptoms include yellowing and mottling of leaves, stunted growth, and reduced yield. The control methods are to use disease-free cuttings, plant resistant varieties, and remove infected plants early.
---
Question: What are the top causes of death in Nigeria?
Answer: The top causes of death in Nigeria are:

- Malaria (20%)
- Lower Respiratory Infection (19%)
- HIV/AIDS (9%)
- Diarrheal Diseases (5%)
- Road Injuries (5%)
- Protein-energy malnutrition
- Cancer
- Meningitis
- Stroke
- Tuberculosis
---
Question: What are the symptoms of Cassava Mosaic Disease?
Answer: The symptoms of Cassava Mosaic Disease are yellowing and mottling of leaves, stunted growth, and reduced yield.
---
Question: What is Mastitis and how is it managed in dairy animals?
Answer: Mastitis is an infection that affects the udder of dairy cattle and goats, characterized by symptoms such as a swollen udder, abnor

Failed to batch ingest runs: LangSmithConnectionError('Connection error caused failure to POST https://api.smith.langchain.com/runs/batch  in LangSmith API. Please confirm your internet connection.. ConnectionError(MaxRetryError(\'HTTPSConnectionPool(host=\\\'api.smith.langchain.com\\\', port=443): Max retries exceeded with url: /runs/batch (Caused by NameResolutionError("HTTPSConnection(host=\\\'api.smith.langchain.com\\\', port=443): Failed to resolve \\\'api.smith.langchain.com\\\' ([Errno 11001] getaddrinfo failed)"))\'))')
Failed to batch ingest runs: LangSmithConnectionError('Connection error caused failure to POST https://api.smith.langchain.com/runs/batch  in LangSmith API. Please confirm your internet connection.. ConnectionError(MaxRetryError(\'HTTPSConnectionPool(host=\\\'api.smith.langchain.com\\\', port=443): Max retries exceeded with url: /runs/batch (Caused by NameResolutionError("HTTPSConnection(host=\\\'api.smith.langchain.com\\\', port=443): Failed to resolve \\\'api.

Step 1: Define Evaluator Prompts

In [15]:
# Prompt for faithfulness
faithfulness_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are an evaluator. Given the CONTEXT and ANSWER, judge if the ANSWER is fully supported by the CONTEXT. Respond with only "yes" or "no".'),
    ('human', 'CONTEXT:\n{context}\n\nANSWER:\n{answer}')
])

# Prompt for answer relevance
relevance_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are an evaluator. Given the QUESTION and ANSWER, judge if the ANSWER directly addresses the QUESTION. Respond with only "yes" or "no".'),
    ('human', 'QUESTION:\n{question}\n\nANSWER:\n{answer}')
])

Step 2: Create Evaluator Functions

In [ ]:
def evaluate_faithfulness(question, answer, docs):
    context = '\n\n'.join(doc.page_content for doc in docs)
    chain = faithfulness_prompt | llm | StrOutputParser()
    result = chain.invoke({'context': context, 'answer': answer})
    return result.strip().lower() == 'yes'
    
def evaluate_answer_relevance(question, answer):
    chain = relevance_prompt | llm | StrOutputParser()
    result = chain.invoke({'question': question, 'answer': answer})
    return result.strip().lower() == 'yes'

Step 3: Run Manual Evaluation

In [22]:
for q in examples:
    # # Skip e-commerce question
    if 'Spacious' in q:
        continue
    
    docs = retriever.invoke(q)
    answer = rag_chain.invoke(q)
    faithful = evaluate_faithfulness(q, answer, docs)
    relevant = evaluate_answer_relevance(q, answer)
    
    print(f'Question: {q}')
    print(f'Answer: {answer}')
    print(f'Faithful: {faithful}')
    print(f'Relevant: {relevant}')
    print('-' * 100)    

Question: What are the common crop diseases and their control methods?
Answer: The common crop disease mentioned is Cassava Mosaic Disease. The affected crop is cassava, and the symptoms include yellowing and mottling of leaves, stunted growth, and reduced yield. The control methods are to use disease-free cuttings, plant resistant varieties, and remove infected plants early.
Faithful: True
Relevant: False
----------------------------------------------------------------------------------------------------
Question: What are the top causes of death in Nigeria?
Answer: The top causes of death in Nigeria are as follows:

- Malaria (20%)
- Lower Respiratory Infection (19%)
- HIV/AIDS (9%)
- Diarrheal Diseases (5%)
- Road Injuries (5%)
- Protein-energy malnutrition
- Cancer
- Meningitis
- Stroke
- Tuberculosis
Faithful: True
Relevant: True
----------------------------------------------------------------------------------------------------
Question: What are the symptoms of Cassava Mosaic Di

**Context Relevance:** This checks if the retrieved chunks are actually relevant to the question.

Add Manual Context Relevance Evaluator

* Define the prompt

In [23]:
context_relevance_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are an evaluator. Given the QUESTION and RETRIEVED CONTEXT, judge if the RETRIEVED CONTEXT is relevant to the QUESTION. Respond with only "yes" or "no".'),
    ('human', 'QUESTION:\n{question}\n\nRETRIEVED CONTEXT:\n{context}')  
])

Define the evaluation function

In [24]:
def evaluate_context_relevance(question, docs):
    context = '\n\n'.join(doc.page_content for doc in docs)
    chain = context_relevance_prompt | llm | StrOutputParser()
    result = chain.invoke({'question': question, 'context': context})
    return result.strip().lower() == 'yes'

Run the full evaluation loop with all three metrics

In [25]:
for q in examples:
    if 'Spacious' in q:
        continue
    
    docs = retriever.invoke(q)
    answer = rag_chain.invoke(q)
    
    faithful = evaluate_faithfulness(q, answer, docs)
    relevant = evaluate_answer_relevance(q, answer)
    context_rel = evaluate_context_relevance(q, docs)
    
    print(f'Question: {q}')
    print(f'Answer: {answer}')
    print(f'Faithful: {faithful}')
    print(f'Relevant: {relevant}')
    print(f'Context Relevant: {context_rel}')
    print('-' * 100)

Question: What are the common crop diseases and their control methods?
Answer: The common crop disease mentioned is Cassava Mosaic Disease. The control methods include:

1. Use disease-free cuttings.
2. Plant resistant varieties.
3. Remove infected plants early.
Faithful: True
Relevant: False
Context Relevant: True
----------------------------------------------------------------------------------------------------
Question: What are the top causes of death in Nigeria?
Answer: The top causes of death in Nigeria are as follows:

- Malaria (20%)
- Lower Respiratory Infection (19%)
- HIV/AIDS (9%)
- Diarrheal Diseases (5%)
- Road Injuries (5%)
- Protein-energy malnutrition
- Cancer
- Meningitis
- Stroke
- Tuberculosis
Faithful: True
Relevant: True
Context Relevant: True
----------------------------------------------------------------------------------------------------
Question: What are the symptoms of Cassava Mosaic Disease?
Answer: The symptoms of Cassava Mosaic Disease are yellowing an